### Operational Supervisor

Creates a single Multi-Agent Supervisor (MAS) for the operational dashboard that
routes across **3 Genie spaces** and **6 Knowledge Assistants**:

| Sub-agent | Type | Source |
|---|---|---|
| `revenue-analytics` | Genie | Revenue & Orders Intelligence space |
| `operations-intelligence` | Genie | Operations Intelligence space |
| `menu-analytics` | Genie | Menu & Safety Intelligence space |
| `inspection-reports` | KA | Food safety inspection PDFs |
| `menu-document-search` | KA | Restaurant menu PDFs |
| `legal-complaints` | KA | Legal complaint case files |
| `regulatory-compliance` | KA | Permits, certifications, FDA |
| `audit-findings` | KA | Financial / operational / supply-chain audits |
| `consultancy-strategy` | KA | Strategic consulting reports |

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG             = dbutils.widgets.get("CATALOG")
SUPERVISOR_ENDPOINT = dbutils.widgets.get("SUPERVISOR_ENDPOINT_NAME")
# KA endpoint names are auto-generated by the KA v2.1 API and resolved from
# uc_state below. They are no longer passed in as Job parameters.

In [ ]:
from databricks.sdk import WorkspaceClient
import json, sys, time

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()
API_BASE = "/api/2.0/multi-agent-supervisors"

##### Resolve Genie space IDs from uc_state

Titles must match the strings used by `stages/genie_spaces.ipynb`.

In [ ]:
GENIE_TITLES = {
    "revenue": f"Revenue & Orders Intelligence ({CATALOG})",
    "ops":     f"Operations Intelligence ({CATALOG})",
    "menu":    f"Menu & Safety Intelligence ({CATALOG})",
}
genie_ids = {}

df = spark.sql(f"""
    SELECT resource_data FROM {CATALOG}._internal_state.resources
    WHERE resource_type = 'genie_spaces'
    ORDER BY created_at DESC
""")
for row in df.collect():
    info = json.loads(row.resource_data)
    title = info.get("title")
    for key, expected in GENIE_TITLES.items():
        if title == expected and key not in genie_ids:
            genie_ids[key] = info.get("space_id")

missing = [k for k in GENIE_TITLES if k not in genie_ids]
if missing:
    raise RuntimeError(
        f"Missing Genie space(s) {missing} in uc_state. Run the genie_spaces stage first."
    )

revenue_genie_id = genie_ids["revenue"]
ops_genie_id     = genie_ids["ops"]
menu_genie_id    = genie_ids["menu"]
print(f"Revenue Genie:   {revenue_genie_id}")
print(f"Operations Genie:{ops_genie_id}")
print(f"Menu Genie:      {menu_genie_id}")

##### Resolve KA serving endpoint names from uc_state

In [ ]:
KA_NAMES = [
    f"{CATALOG}-inspection-knowledge",
    f"{CATALOG}-menu-knowledge",
    f"{CATALOG}-legal",
    f"{CATALOG}-regulatory",
    f"{CATALOG}-audits",
    f"{CATALOG}-consultancy",
]


def _ka_endpoint_from_tile_id(tile_id: str) -> str:
    # Databricks serving endpoint provisioned for a KA is named
    # ka-{first-8-chars-of-tile_id}-endpoint. The `endpoint_name` field
    # returned by the v2.1 KA API is a different (display-style) value
    # and MUST NOT be used here — MAS validates the serving-endpoint name.
    if not tile_id or len(tile_id) < 8:
        raise ValueError(f"Invalid KA tile_id for endpoint construction: {tile_id!r}")
    return f"ka-{tile_id[:8]}-endpoint"


ka_tile_ids = {}

try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'knowledge_assistants'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = json.loads(row.resource_data)
        name = info.get("name", "")
        tid  = info.get("tile_id", "")
        if name in KA_NAMES and tid and name not in ka_tile_ids:
            ka_tile_ids[name] = tid
    print(f"uc_state: resolved {len(ka_tile_ids)}/{len(KA_NAMES)} KA tile_ids")
except Exception as e:
    print(f"\u26a0\ufe0f uc_state lookup failed: {e}")

missing_names = [n for n in KA_NAMES if n not in ka_tile_ids]
if missing_names:
    print(f"{len(missing_names)} KA tile_id(s) missing from uc_state \u2014 listing KAs via v2.1 API...")
    ka_params = {}
    while True:
        resp = w.api_client.do("GET", "/api/2.1/knowledge-assistants", query=ka_params)
        for ka in resp.get("knowledge_assistants", []):
            dn  = ka.get("display_name", "")
            kid = ka.get("id", "")
            if dn in missing_names and kid:
                ka_tile_ids[dn] = kid
                missing_names.remove(dn)
        token = resp.get("next_page_token")
        if not token or not missing_names:
            break
        ka_params = {"page_token": token}

if missing_names:
    raise RuntimeError(
        f"Could not resolve tile_ids for: {missing_names}. "
        f"Ensure the knowledge_agents stage ran first."
    )

ka_endpoints = {name: _ka_endpoint_from_tile_id(tid) for name, tid in ka_tile_ids.items()}

inspect_ep_id = ka_endpoints[f"{CATALOG}-inspection-knowledge"]
menu_ep_id    = ka_endpoints[f"{CATALOG}-menu-knowledge"]
legal_ep_id   = ka_endpoints[f"{CATALOG}-legal"]
reg_ep_id     = ka_endpoints[f"{CATALOG}-regulatory"]
audit_ep_id   = ka_endpoints[f"{CATALOG}-audits"]
consult_ep_id = ka_endpoints[f"{CATALOG}-consultancy"]

print(f"Inspection KA endpoint:  {inspect_ep_id}")
print(f"Menu KA endpoint:        {menu_ep_id}")
print(f"Legal KA endpoint:       {legal_ep_id}")
print(f"Regulatory KA endpoint:  {reg_ep_id}")
print(f"Audit KA endpoint:       {audit_ep_id}")
print(f"Consultancy KA endpoint: {consult_ep_id}")

##### Create the Operational Supervisor

In [ ]:
AGENT_NAME = f"{CATALOG}-operational-supervisor"

EXAMPLES = [
    {
        "question": "Give me the executive briefing on the state of the business",
        "guideline": "Should summarize revenue performance, top operational risks, any critical legal or compliance issues, and one strategic recommendation. Keep it to 5 bullet points maximum.",
    },
    {
        "question": "Which location is most at risk right now?",
        "guideline": "Should consider complaint rate, inspection score, legal exposure, and audit findings for each location. Rank and explain the top risk with specific numbers.",
    },
    {
        "question": "Which location has the highest order cancellation rate right now?",
        "guideline": "Must name exactly one specific location with the highest rate. Must include the numeric cancellation rate as a percentage. Routes to revenue-analytics or operations-intelligence.",
    },
    {
        "question": "How does revenue compare across our locations this week?",
        "guideline": "Must provide revenue figures or ranking for multiple locations. Must reference the current week. Routes to revenue-analytics.",
    },
    {
        "question": "Which brand is generating the most revenue right now?",
        "guideline": "Must name a specific brand (not a location). Must include a revenue figure or ranking. Routes to revenue-analytics.",
    },
    {
        "question": "Which location needs the most operational attention right now?",
        "guideline": "Must name one specific location with the highest operational risk. Must justify with at least two operational metrics. Routes to operations-intelligence.",
    },
    {
        "question": "What happened during the Chicago food safety inspection? Were there critical violations?",
        "guideline": "Must reference a specific inspection report by date or ID. Must state the inspection score and/or grade. Must explicitly state whether critical violations were found and cite at least one specific violation code if they exist. Routes to inspection-reports.",
    },
    {
        "question": "Which menu items are gluten-free and under $15?",
        "guideline": "Must list specific items with brand, price, and allergen status. Routes to menu-analytics or menu-document-search.",
    },
    {
        "question": "Compare protein content across all burgers in our network",
        "guideline": "Must return per-item protein values across brands. Routes to menu-analytics.",
    },
    {
        "question": "Do we have any active high-risk legal cases? What is the total financial exposure?",
        "guideline": "Must confirm whether active cases exist, cite at least one specific case number (CK-XX-XXXX), include a risk classification, state a financial exposure amount, and remind users to involve legal counsel. Routes to legal-complaints.",
    },
    {
        "question": "Are there any permits or regulatory certificates expiring in the next 60 days?",
        "guideline": "Must list specific document IDs or permit names with expiry dates and locations. Routes to regulatory-compliance.",
    },
    {
        "question": "What were the most significant audit findings this quarter?",
        "guideline": "Must cite the auditing firm and audit period, classify findings by severity, and state remediation status. Routes to audit-findings.",
    },
    {
        "question": "What do our consultants recommend as the top AI investments for the next 90 days?",
        "guideline": "Must reference a specific consulting report, include at least one concrete recommendation with a financial metric, and frame in the 90-day horizon. Routes to consultancy-strategy.",
    },
    {
        "question": "Give me a board deck summary: revenue performance, top operational risk, legal exposure, and one strategic recommendation",
        "guideline": "Must explicitly address all four domains. Must include at least one concrete number per domain. Must be structured with clear sections. Routes to multiple agents.",
    },
]

SUPERVISOR_BODY = {
    "name": AGENT_NAME,
    "description": (
        "Operational AI assistant for Casper's Kitchens. Synthesises intelligence across revenue "
        "analytics, operations, food safety, menu and nutrition data, legal exposure, regulatory "
        "compliance, audit findings, and strategic consulting across all 8 ghost kitchen locations "
        "(US: San Francisco, Silicon Valley, Bellevue, Chicago; EMEA: London, Munich, Amsterdam, "
        "Vianen) and 16 restaurant brands."
    ),
    "endpoint_name": SUPERVISOR_ENDPOINT,
    "agents": [
        {
            "agent_type": "genie",
            "genie_space": {"id": revenue_genie_id},
            "name": "revenue-analytics",
            "description": (
                "ONLY call for questions about: revenue ($), sales totals, order counts, average order "
                "value, brand sales rankings, location revenue comparisons, or financial performance "
                "metrics. Keywords: revenue, sales, earned, orders, top performing, best/worst location "
                "by revenue, avg order, financial. Do NOT call for food safety, inspections, legal, "
                "audits, menus, or strategic questions."
            ),
        },
        {
            "agent_type": "genie",
            "genie_space": {"id": ops_genie_id},
            "name": "operations-intelligence",
            "description": (
                "ONLY call for questions about: order throughput, kitchen operations, delivery performance, "
                "food safety inspection scores, food safety violation counts, location health metrics, "
                "or operational risk. Keywords: operations, kitchen, delivery, inspection score, food "
                "safety grade, throughput, busiest hours, peak demand, operational issues, cancel rate, "
                "complaint rate. Do NOT call for revenue totals, legal cases, audit reports, or strategy. "
                "IMPORTANT: 'complaints' here means customer operational complaints — NOT legal filings."
            ),
        },
        {
            "agent_type": "genie",
            "genie_space": {"id": menu_genie_id},
            "name": "menu-analytics",
            "description": (
                "ONLY call for structured queries about menu items, nutrition, allergens, item pricing "
                "tiers, brand-level menu comparisons, or per-location compliance summaries. Keywords: "
                "calories, protein, fat, carbs, allergen-free, gluten-free, dairy-free, item price, "
                "brand menu, nutrition comparison, healthy options. Do NOT call for descriptive "
                "questions about individual dishes — use menu-document-search for those."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": inspect_ep_id},
            "name": "inspection-reports",
            "description": (
                "ONLY call for questions about specific food safety inspection documents, detailed "
                "inspector findings, corrective action details, or historical inspection report text. "
                "For inspection SCORES/GRADES use operations-intelligence instead."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": menu_ep_id},
            "name": "menu-document-search",
            "description": (
                "ONLY call for descriptive questions about specific dishes — ingredient details, "
                "preparation, dish description, or specific menu content. For nutrition stats, "
                "allergen filtering, or price comparisons, use menu-analytics instead."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": legal_ep_id},
            "name": "legal-complaints",
            "description": (
                "ONLY call for questions about lawsuits, legal cases, customer complaint filings, "
                "litigation status, legal liability, legal issues, or specific complaint case numbers. "
                "Always surface case numbers (format CK-XX-XXXX), risk levels (HIGH/MEDIUM/LOW), "
                "and amounts at stake."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": reg_ep_id},
            "name": "regulatory-compliance",
            "description": (
                "ONLY call for questions about permits, licenses, certifications, regulatory compliance "
                "documents, or specific regulatory requirements."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": audit_ep_id},
            "name": "audit-findings",
            "description": (
                "ONLY call for questions about financial or operational audit reports, audit findings, "
                "auditor recommendations, or specific audit references."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": consult_ep_id},
            "name": "consultancy-strategy",
            "description": (
                "ONLY call for questions about strategic recommendations, consultant advice, improvement "
                "strategies, or specific consultancy reports."
            ),
        },
    ],
    "instructions": (
        "You are an elite operational AI assistant for Casper's Kitchens, a ghost kitchen network "
        "operating 16 restaurant brands across 8 locations (US: San Francisco, Silicon Valley, Bellevue, "
        "Chicago; EMEA: London, Munich, Amsterdam, Vianen). Synthesise intelligence from multiple "
        "sources to deliver concise, executive-ready insights. Always specify which location(s) you "
        "are referencing. Lead with the most important finding, then provide supporting detail.\n\n"
        "DATA SOURCE BOUNDARIES:\n"
        "(1) 'Legal issues', 'lawsuits', 'legal cases', 'legal exposure' → legal-complaints agent "
        "(CK-XX-XXXX case numbers), NOT food safety violations or operational complaints.\n"
        "(2) 'Food safety violations' → operations-intelligence (counts/grades) or inspection-reports "
        "(detailed findings) — they are compliance issues, not lawsuits.\n"
        "(3) 'Customer complaints' (complaint rate, cancel rate) → operations-intelligence — they are "
        "operational metrics, not legal filings.\n"
        "(4) 'Allergens / nutrition / prices' → menu-analytics (structured) or menu-document-search "
        "(descriptive dish details).\n"
        "Never conflate food safety violations or operational complaint counts with legal case filings. "
        "When discussing legal or audit matters, remind users to involve counsel or auditors for "
        "decisions. Be direct, data-driven, and strategic."
    ),
}

In [ ]:
def _list_mas():
    mas, params = [], {}
    try:
        while True:
            resp = w.api_client.do("GET", API_BASE, query=params)
            mas.extend(resp.get("multi_agent_supervisors", []))
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"  MAS list error: {e}")
    return mas


def get_tile_by_name(name):
    try:
        params = {}
        while True:
            resp = w.api_client.do("GET", "/api/2.0/tiles", query=params)
            for tile in resp.get("tiles", []):
                if tile.get("name") == name:
                    return tile
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception:
        pass
    return None


agent_id = None
is_new = False

try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'multi_agent_supervisors'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = json.loads(row.resource_data)
        if (info.get("endpoint_name") == SUPERVISOR_ENDPOINT or info.get("name") == AGENT_NAME) and info.get("tile_id"):
            agent_id = info["tile_id"]
            print(f"\u267b\ufe0f Supervisor found in uc_state ({agent_id}) — skipping create")
            break
except Exception as e:
    print(f"\u26a0\ufe0f uc_state lookup failed: {e}")

if not agent_id:
    for item in _list_mas():
        mas = item.get("multi_agent_supervisor", item)
        if mas.get("endpoint_name") == SUPERVISOR_ENDPOINT or mas.get("name") == AGENT_NAME:
            agent_id = item.get("tile_id") or mas.get("tile_id") or mas.get("id")
            print(f"\u267b\ufe0f Supervisor found in MAS API ({agent_id}) — skipping create")
            break

if not agent_id:
    try:
        w.api_client.do("POST", API_BASE, body=SUPERVISOR_BODY)
        print("\u2705 Created Operational Supervisor")
        is_new = True
    except Exception as e:
        if "already exists" in str(e).lower():
            print("\u267b\ufe0f Supervisor already exists, proceeding")
        else:
            raise
    tile = get_tile_by_name(AGENT_NAME)
    if tile:
        agent_id = tile["tile_id"]
        print(f"Resolved tile_id: {agent_id}")
    else:
        agent_id = AGENT_NAME
        print(f"\u26a0\ufe0f Using name as fallback: {agent_id}")

if isinstance(agent_id, str) and len(agent_id) >= 8 and "-" in agent_id:
    actual_endpoint_name = f"mas-{agent_id[:8]}-endpoint"
else:
    actual_endpoint_name = SUPERVISOR_ENDPOINT
print(f"MAS serving endpoint name: {actual_endpoint_name}")

add(CATALOG, "multi_agent_supervisors", {
    "endpoint_name": actual_endpoint_name,
    "tile_id": agent_id,
    "name": AGENT_NAME,
})

# ── Drift recovery: PATCH the supervisor with fresh sub-agent IDs ────────────
# The `genie_spaces` stage is non-idempotent and creates fresh space IDs on
# every deploy. When this stage reuses an existing supervisor (uc_state hit
# above), its wired Genie IDs / KA endpoints can point at trashed resources,
# which surfaces in chat as "permissions issue accessing the revenue analytics
# system" (it's actually a 404 on the trashed Genie space). Re-applying
# SUPERVISOR_BODY against the existing tile id keeps the supervisor in lock-
# step with the current resources. No-op when nothing drifted.
#
# Hard failure on exhausted retries: this used to log a warning and continue,
# which left the supervisor pointing at trashed resources and broke the demo
# chat silently.  Now we retry the PATCH 3x with exponential backoff (1/2/4 s)
# and raise on the final failure so the deploy aborts with a clear error
# instead of leaving a half-wired supervisor in place.
import time as _time

if not is_new and isinstance(agent_id, str) and "-" in agent_id and len(agent_id) >= 8:
    existing = w.api_client.do("GET", f"{API_BASE}/{agent_id}")
    wired = (existing.get("multi_agent_supervisor") or {}).get("agents") or []

    def _key(a):
        t = a.get("agent_type", "")
        if t in ("genie", "genie-space"):
            return ("genie", a.get("name"), (a.get("genie_space") or {}).get("id"))
        if t in ("ka", "knowledge-assistant"):
            return ("ka", a.get("name"), (a.get("serving_endpoint") or {}).get("name"))
        return (t, a.get("name"))

    wired_keys  = {_key(a) for a in wired}
    target_keys = {_key(a) for a in SUPERVISOR_BODY["agents"]}

    if wired_keys == target_keys:
        print("✅ Supervisor wiring matches uc_state — no PATCH needed")
    else:
        drifted = target_keys ^ wired_keys
        print(f"⚠️  Supervisor wiring drift detected ({len(drifted)} mismatched sub-agents) — PATCHing…")
        _last_err = None
        for _attempt in range(1, 4):
            try:
                w.api_client.do("PATCH", f"{API_BASE}/{agent_id}", body=SUPERVISOR_BODY)
                print(f"✅ Supervisor PATCHed with current Genie/KA wiring (attempt {_attempt}/3)")
                _last_err = None
                break
            except Exception as e:
                _last_err = e
                _wait = 2 ** (_attempt - 1)  # 1s, 2s, 4s
                print(f"⚠️  Drift-recovery PATCH attempt {_attempt}/3 failed: {e!r}")
                if _attempt < 3:
                    print(f"   retrying in {_wait}s…")
                    _time.sleep(_wait)
        if _last_err is not None:
            # Re-raise so the stage fails loudly.  A stale supervisor breaks
            # the demo chat ("permissions issue accessing the revenue
            # analytics system") in a way that's expensive to diagnose at
            # demo time — we'd much rather fail the deploy here.
            raise RuntimeError(
                f"Supervisor drift-recovery PATCH failed after 3 attempts: {_last_err!r}. "
                f"The supervisor at tile_id={agent_id} is still wired to stale Genie/KA "
                f"resources.  Re-run this stage once the Databricks API is reachable, or "
                f"delete the supervisor from uc_state to force re-creation."
            ) from _last_err

##### Wait for endpoint, then attach evaluation examples

Only runs on first creation; on re-runs, the supervisor is reused and examples are
left in place. Endpoint readiness is also re-polled by `Readiness_Check`.

In [ ]:
if is_new and agent_id and agent_id != AGENT_NAME and actual_endpoint_name:
    print(f"\nPolling MAS endpoint readiness ({actual_endpoint_name})...")
    ep_ready = False
    for attempt in range(1, 61):
        try:
            resp = w.api_client.do("GET", f"/api/2.0/serving-endpoints/{actual_endpoint_name}")
            state = resp.get("state", {})
            ready = str(state.get("ready", "")).upper()
            if ready == "READY":
                print(f"  \u2705 Endpoint READY")
                ep_ready = True
                break
            print(f"  [{attempt}/60] state={ready} — waiting 20s…")
        except Exception as e:
            print(f"  [{attempt}/60] poll error: {e} — waiting 20s…")
        time.sleep(20)

    if ep_ready:
        try:
            w.api_client.do(
                "POST",
                f"/api/2.0/tiles/{agent_id}/examples",
                body={"examples": EXAMPLES},
            )
            print(f"\u2705 Examples added ({len(EXAMPLES)} questions)")
        except Exception as e:
            print(f"\u26a0\ufe0f  Could not add examples: {e}")
    else:
        print("\u26a0\ufe0f  Endpoint not READY after 20 min — skipping examples (re-run to retry)")
else:
    print("\u2705 Supervisor already running — skipping poll and examples")

print(f"\n\u2705 Operational Supervisor stage complete")
print(f"   Endpoint: {actual_endpoint_name}")
print("   Sub-agents: revenue-analytics, operations-intelligence, menu-analytics, "
      "inspection-reports, menu-document-search, legal-complaints, regulatory-compliance, "
      "audit-findings, consultancy-strategy")